<a href="https://colab.research.google.com/github/dohee-jin/data-mining-final-project/blob/main/notebooks/kbo_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 골든글러브 후보 예측

- 선수 개인 기록을 활용해 2026년 골든글러브 후보 가능성이 높은 선수를 예측한다.
- 타자와 투수는 기록 컬럼이 다르므로 분리해서 분석한다.

---

## 2. 골든글로브 후보 예측 데이터 사용 전략

데이터는 크게 세 종류로 나눈다.

| 구분 | 출처 | 직접 정리 여부 | 사용 목적 |
|---|---|---:|---|
| 과거 선수 기록 | Kaggle KBO Player Dataset 1982-2025 | X | 골든글러브 예측 학습 데이터 |
| 2026 현재 선수 기록 | KBO 공식 홈페이지 기록실 | O | 2026 골든글러브 후보 예측 대상 |
| 골든글러브 수상자 목록 | KBO 공식 골든글러브 수상 현황 | O | `is_goldenglove` 라벨 생성 |

---

## 3. 골든글러브 후보 예측용 데이터

### 3.1 Kaggle에서 가져올 데이터

Kaggle의 `KBO Player Dataset (1982-2025)`를 과거 학습 데이터로 사용한다.

사용할 데이터는 다음과 같다.

```text
kbo_batting_stats_by_season_1982-2025.csv
kbo_pitching_stats_by_season_1982-2025.csv
```

Kaggle 데이터는 1982년부터 2025년까지의 KBO 정규시즌 선수 타격 및 투수 기록을 포함한다. 실제 분석에서는 너무 오래된 데이터까지 모두 쓰기보다, 최근 야구 환경과 비슷한 기간만 필터링하여 사용한다.

추천 사용 기간:

```text
2015~2025
```

### 3.2 직접 정리해야 하는 2026 선수 기록

2026년은 아직 시즌이 진행 중이므로 Kaggle 과거 데이터에 포함되어 있지 않다. 따라서 KBO 공식 홈페이지에서 현재 기록을 복사해 CSV로 직접 정리하고 깃허브에 csv로 raw 파일을 업로드했다.

```text
data/raw/kbo_hitter_data_2026.csv
data/raw/kbo_pitcher_data_2026.csv
```

2026 선수 기록 파일에는 반드시 `year` 컬럼을 추가한다.   
타자 기록에는 골든글러브 포지션별 예측을 위해 `position` 칼럼을 추가한다.   
일부 선수는 시즌 중 여러 포지션에 출전하므로, 본 분석에서는 데이터 단순화와 모델 적용의 일관성을 위해 선수별 대표 포지션을 하나만 부여하였다. 대표 포지션은 KBO 기록실의 선수 구분 또는 주요 출전 포지션을 기준으로 정리하였다.

```text
year = 2026
position = C, 1B, 2B, 3B, SS, OF, DH
```

예시:

| year | player | team | avg | hr | rbi | ops |
|---:|---|---|---:|---:|---:|---:|
| 2026 | 선수명 | LG | 0.315 | 12 | 45 | 0.890 |

---

### 4. kaggle 데이터셋 접근 불가
kaggle 데이터셋이 내려가 접근하지 못해 해당 데이터셋으로의 모델 생성, 예측은 불가능하게 됨.  
`https://huggingface.co/datasets/juhonov/KBOresearch?utm_source=chatgpt.com` 의 선수 기록 수집기를 통해 2000-2025년도 선수 기록을 바탕으로 데이터를 수집, 학습하기로 계획을 변경함

### kagglehub로 kbo 1982-2025 데이터셋 불러오기

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("netsong/kbo-player-dataset-by-regular-season-1982-2025")

print("Path to dataset files:", path)

KaggleApiHTTPError: 403 Client Error.

You don't have permission to access resource at URL: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDataset. Please make sure you are authenticated if you are trying to access a private resource or a resource requiring consent.

In [5]:
import os
import pandas as pd

path = kagglehub.dataset_download("netsong/kbo-player-dataset-by-regular-season-1982-2025")

for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

KaggleApiHTTPError: 403 Client Error.

You don't have permission to access resource at URL: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDataset. Please make sure you are authenticated if you are trying to access a private resource or a resource requiring consent.

## 다른 데이터셋 사용

In [11]:
!pip install -q datasets

from datasets import load_dataset
from huggingface_hub import hf_hub_download
import pandas as pd

# Hugging Face에서 JSON 파일 직접 다운로드
hitter_file = hf_hub_download(
    repo_id="juhonov/KBOresearch",
    filename="kbo_hitter_stats_2000_2025.json",
    repo_type="dataset"
)

pitcher_file = hf_hub_download(
    repo_id="juhonov/KBOresearch",
    filename="kbo_pitcher_stats_2000_2025.json",
    repo_type="dataset"
)

print(hitter_file)
print(pitcher_file)

/root/.cache/huggingface/hub/datasets--juhonov--KBOresearch/snapshots/14fd8e53426a5e3bd21b576fa109ea8928463008/kbo_hitter_stats_2000_2025.json
/root/.cache/huggingface/hub/datasets--juhonov--KBOresearch/snapshots/14fd8e53426a5e3bd21b576fa109ea8928463008/kbo_pitcher_stats_2000_2025.json


In [14]:
# Json 구조 확인
import json

with open(hitter_file, "r", encoding="utf-8") as f:
    hitter_json = json.load(f)

with open(pitcher_file, "r", encoding="utf-8") as f:
    pitcher_json = json.load(f)

print(type(hitter_json))
print(type(pitcher_json))

if isinstance(hitter_json, dict):
    print(hitter_json.keys())

if isinstance(pitcher_json, dict):
    print(pitcher_json.keys())

<class 'dict'>
<class 'dict'>
dict_keys(['metadata', 'data'])
dict_keys(['metadata', 'data'])


In [16]:
# json 데이터 구조 확인
def inspect_json(obj, name="root"):
    print(f"\n{name}: type={type(obj)}")

    if isinstance(obj, dict):
        print("keys:", list(obj.keys()))
        for k, v in obj.items():
            print(f"  {k}: {type(v)}", end="")
            if isinstance(v, list):
                print(f", list length={len(v)}")
                if len(v) > 0:
                    print("    first item type:", type(v[0]))
                    print("    first item:", v[0])
            elif isinstance(v, dict):
                print(f", dict keys={list(v.keys())[:10]}")
            else:
                print(f", value={v}")

    elif isinstance(obj, list):
        print("list length:", len(obj))
        if len(obj) > 0:
            print("first item type:", type(obj[0]))
            print("first item:", obj[0])

inspect_json(hitter_json, "hitter_json")
inspect_json(pitcher_json, "pitcher_json")


hitter_json: type=<class 'dict'>
keys: ['metadata', 'data']
  metadata: <class 'dict'>, dict keys=['description', 'source', 'collected_at', 'total_records']
  data: <class 'list'>, list length=8188
    first item type: <class 'dict'>
    first item: {'순위': '1', 'playerId': '62558', '선수명': '김준태', '팀명': 'LG', 'AVG': '1.000', 'G': '2', 'PA': '2', 'AB': '1', 'R': '1', 'H': '1', '2B': '0', '3B': '0', 'HR': '0', 'TB': '1', 'RBI': '0', 'SAC': '0', 'SF': '0', 'year': 2025, 'teamCode': 'LG', 'teamName': 'LG'}

pitcher_json: type=<class 'dict'>
keys: ['metadata', 'data']
  metadata: <class 'dict'>, dict keys=['description', 'source', 'collected_at', 'total_records']
  data: <class 'list'>, list length=5696
    first item type: <class 'dict'>
    first item: {'순위': '1', 'playerId': '50904', '선수명': '이종준', '팀명': 'LG', 'ERA': '0.00', 'G': '2', 'W': '0', 'L': '0', 'SV': '0', 'HLD': '0', 'WPCT': '-', 'IP': '1 2/3', 'H': '0', 'HR': '0', 'BB': '1', 'HBP': '0', 'SO': '2', 'R': '0', 'ER': '0', 'WHIP': '0

In [22]:
from google.colab import files

# 데이터프레임 생성
hitter = pd.DataFrame(hitter_json["data"])
pitcher = pd.DataFrame(pitcher_json["data"])

print(hitter.shape)
print(pitcher.shape)

display(hitter.head())
display(pitcher.head())

# 데이터프레임 로컬 저장
# 로컬 저장 후 깃허브에 직접 업로드 필요
"""
hitter.to_csv("kbo_hitter_stats_2000-2025.csv", index=False, encoding="utf-8-sig")
pitcher.to_csv("kbo_pitcher_stats_2000-2025.csv", index=False, encoding="utf-8-sig")

files.download("kbo_hitter_stats_2000-2025.csv")
files.download("kbo_pitcher_stats_2000-2025.csv")
"""

(8188, 27)
(5696, 26)


,순위,playerId,선수명,팀명,AVG,G,PA,AB,R,H,...,year,teamCode,teamName,SB,CS,BB,HBP,SO,GDP,E
0,1,62558,김준태,LG,1.000,2,2,1,1,1,...,2025,LG,LG,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,54166,김현종,LG,0.400,10,6,5,3,2,...,2025,LG,LG,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,65207,신민재,LG,0.313,135,538,463,87,145,...,2025,LG,LG,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,53123,오스틴,LG,0.313,116,499,425,82,133,...,2025,LG,LG,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,68119,문성주,LG,0.305,135,542,475,57,145,...,2025,LG,LG,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,순위,playerId,선수명,팀명,ERA,G,W,L,SV,HLD,...,SO,R,ER,WHIP,year,teamCode,teamName,CG,SHO,TBF
0,1,50904,이종준,LG,0.00,2,0,0,0,0,...,2,0,0,0.60,2025,LG,LG,NaN,NaN,NaN
1,2,77263,김강률,LG,1.46,12,1,0,1,4,...,9,2,2,1.22,2025,LG,LG,NaN,NaN,NaN
2,3,61145,이우찬,LG,1.89,23,0,1,0,0,...,20,6,4,1.42,2025,LG,LG,NaN,NaN,NaN
3,4,55167,김영우,LG,2.40,66,3,2,1,7,...,56,17,16,1.32,2025,LG,LG,NaN,NaN,NaN
4,5,50106,유영찬,LG,2.63,39,2,2,21,1,...,52,14,12,1.32,2025,LG,LG,NaN,NaN,NaN


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
# hitter = pd.read_csv("/kaggle/input/kbo-player-dataset-by-regular-season-1982-2025/kbo_batting_stats_by_season_1982-2025.csv")
# pitcher = pd.read_csv("/kaggle/input/kbo-player-dataset-by-regular-season-1982-2025/kbo_pitching_stats_by_season_1982-2025.csv")

hitter.head()
pitcher.head()

# 2015~2025년 데이터만 필터링
hitter_recent = hitter[hitter["year"] > 2015]
pitcher_recent = pitcher[pitcher["year"] > 2015]

hitter_recent.head()
pitcher_recent.head()

,순위,playerId,선수명,팀명,ERA,G,W,L,SV,HLD,...,SO,R,ER,WHIP,year,teamCode,teamName,CG,SHO,TBF
0,1,50904,이종준,LG,0.00,2,0,0,0,0,...,2,0,0,0.60,2025,LG,LG,NaN,NaN,NaN
1,2,77263,김강률,LG,1.46,12,1,0,1,4,...,9,2,2,1.22,2025,LG,LG,NaN,NaN,NaN
2,3,61145,이우찬,LG,1.89,23,0,1,0,0,...,20,6,4,1.42,2025,LG,LG,NaN,NaN,NaN
3,4,55167,김영우,LG,2.40,66,3,2,1,7,...,56,17,16,1.32,2025,LG,LG,NaN,NaN,NaN
4,5,50106,유영찬,LG,2.63,39,2,2,21,1,...,52,14,12,1.32,2025,LG,LG,NaN,NaN,NaN


### Github에 업로드 한 2026 raw 데이터 불러오기


In [27]:
golden_glove = pd.read_csv("https://raw.githubusercontent.com/dohee-jin/data-mining-final-project/main/data/raw/kbo_goldenglove_data_2025.csv")
hitter_2026 = pd.read_csv("https://raw.githubusercontent.com/dohee-jin/data-mining-final-project/main/data/raw/kbo_hitter_data_2026.csv")
pitcher_2026 = pd.read_csv("https://raw.githubusercontent.com/dohee-jin/data-mining-final-project/main/data/raw/kbo_pitcher_data_2026.csv")

golden_glove.head()
hitter_2026.head()
pitcher_2026.head()

,rank,name,team,ERA,G,W,L,SV,HLB,WPCT,...,NP,AVG,2B,3B,SAC,SF,IBB,WP,BK,year
0,1,후라도,삼성,2.61,12,3,1,0,0,0.750,...,1156,0.259,6,2,4,3,0,4,0,2026
1,2,올러,KIA,2.66,13,7,5,0,0,0.583,...,1211,0.182,5,2,5,1,0,3,0,2026
2,3,류현진,한화,2.84,12,8,2,0,0,0.800,...,1059,0.239,14,1,7,2,1,1,0,2026
3,4,최민석,두산,2.88,12,6,2,0,0,0.750,...,1098,0.215,13,0,3,2,0,3,1,2026
4,5,알칸타라,키움,3.12,12,6,4,0,0,0.600,...,1154,0.259,12,2,1,1,0,2,0,2026


## 데이터 전처리
### 1. Kaggle 데이터와 KBO 2026 데이터 칼럼명 맞추기

#### 기록용어 참고
01. 타자 기록
  - 2B: 2루타
  - 3B: 3루타
  - AB: 타수
  - AO: 뜬공
  - AVG: 타율
  - BB: 볼넷
  - BB/K: 볼넷/삼진
  - CS: 도루실패
  - E: 실책
  - G: 경기
  - GDP: 병살타
  - GO : 땅볼
  - GO/AO : 땅볼/뜬공
  - GPA : (1.8x출루율+장타율)/4
  - GW RBI : 결승타
  - H : 안타
  - HBP(HP) : 사구
  - HR : 홈런
  - IBB(IB) : 고의4구
  - ISOP : 순수장타율
  - MH : 멀티히트
  - OBP : 출루율
  - OPS : 출루율+장타율
  - P/PA : 투구수/타석
  - PA : 타석
  - PH-BA : 대타타율
  - R : 득점
  - RBI : 타점
  - RISP : 득점권타율
  - SAC(SH) : 희생번트
  - SB : 도루
  - SF : 희생플라이
  - SLG : 장타율
  - SO : 삼진
  - TB: 루타
  - XBH : 장타
  - XR : 추정득점

02. 투수기록
  - 2B : 2루타
  - 3B : 3루타
  - AO : 뜬공
  - AVG : 피안타율
  - BABIP : 인플레이타구타율
  - BB : 볼넷
  - BB/9 : 9이닝당 볼넷
  - BK : 보크
  - BSV : 블론세이브
  - CG : 완투
  - ER : 자책점
  - ERA : 평균자책점
  - G : 경기
  - GDP : 병살타
  - GF : 종료
  - GO : 땅볼
  - GO/AO : 땅볼/뜬공
  - GS : 선발
  - H : 피안타
  - HBP(HP) : 사구
  - HLD(HD) : 홀드
  - HR : 홈런
  - IBB(IB) : 고의4구
  - IP : 이닝
  - K/9 : 9이닝당 삼진
  - K/BB : 삼진/볼넷
  - L : 패
  - NP : 투구수
  - OBP : 피출루율
  - OPS : 피출루율+피장타율
  - P/G : 투구수/경기
  - P/IP : 투구수/이닝
  - QS : 퀄리티스타트
  - R : 실점
  - SAC(SH) : 희생번트
  - SF : 희생플라이
  - SHO : 완봉
  - SLG : 피장타율
  - SO : 삼진
  - SV(S) : 세이브
  - SVO : 세이브기회
  - TBF : 타자수
  - TS : 터프세이브
  - W : 승
  - Wgr : 구원승
  - Wgs : 선발승
  - WHIP : 이닝당 출루허용률
  - WP : 폭투
  - WPCT : 승률

####
   

In [28]:
# 각 데이터의 칼럼명 확인
print(hitter_recent.columns)
print(pitcher_recent.columns)
print(hitter_2026.columns)
print(pitcher_2026.columns)
print(golden_glove.columns)


Index(['순위', 'playerId', '선수명', '팀명', 'AVG', 'G', 'PA', 'AB', 'R', 'H', '2B',
       '3B', 'HR', 'TB', 'RBI', 'SAC', 'SF', 'year', 'teamCode', 'teamName',
       'SB', 'CS', 'BB', 'HBP', 'SO', 'GDP', 'E'],
      dtype='object')
Index(['순위', 'playerId', '선수명', '팀명', 'ERA', 'G', 'W', 'L', 'SV', 'HLD',
       'WPCT', 'IP', 'H', 'HR', 'BB', 'HBP', 'SO', 'R', 'ER', 'WHIP', 'year',
       'teamCode', 'teamName', 'CG', 'SHO', 'TBF'],
      dtype='object')
Index(['rank', 'name', 'team', 'position', 'AVG', 'G', 'PA', 'AB', 'R', 'H',
       '2B', '3B', 'HR', 'TB', 'RBI', 'SAC', 'SF', 'BB', 'IBB', 'HBP', 'SO',
       'GDP', 'SLG', 'OBP', 'OPS', 'MH', 'RISP', 'PH-BA', 'year'],
      dtype='object')
Index(['rank', 'name', 'team', 'ERA', 'G', 'W', 'L', 'SV', 'HLB', 'WPCT', 'IP',
       'H', 'HR', 'BB', 'HBP', 'SO', 'R', 'ER', 'WHIP', 'CG', 'SHO', 'QS',
       'BSV', 'TBF', 'NP', 'AVG', '2B', '3B', 'SAC', 'SF', 'IBB', 'WP', 'BK',
       'year'],
      dtype='object')
Index(['year', 'P', 'C', '1B', '2

In [50]:
# 타자 데이터 컬럼명 정리(Kaggle 기준으로 정리)
"""
hitter_2026 = hitter_2026.rename(columns={
    "name": "Name",
    "team": "Team",
    "year": "Year",
    "position": "Pos",
    "HBP": "HP",
    "IBB": "IB",
    "SAC": "SH"
})
"""
# 2026 데이터 기준으로 정리
hitter_recent = hitter_recent.rename(columns = {
    "순위": "rank",
    "선수명": "name",
    "팀명": "team",
})

pitcher_recent = pitcher_recent.rename(columns = {
    "순위": "rank",
    "선수명": "name",
    "팀명": "team",
})

# Strip whitespace from name and team columns
hitter_recent["name"] = hitter_recent["name"].str.strip()
hitter_recent["team"] = hitter_recent["team"].str.strip()
pitcher_recent["name"] = pitcher_recent["name"].str.strip()
pitcher_recent["team"] = pitcher_recent["team"].str.strip()

# 투수 데이터 컬럼명 정리(Kaggle 기준으로 정리)
"""
pitcher_2026 = pitcher_2026.rename(columns={
    "name": "Name",
    "team": "Team",
    "year": "Year",
    "HLB": "HD",
    "SV": "S"
})
"""

print(hitter_2026.columns)
print(pitcher_2026.columns)
print(hitter_recent.columns)
print(pitcher_recent.columns)

Index(['rank', 'name', 'team', 'position', 'AVG', 'G', 'PA', 'AB', 'R', 'H',
       '2B', '3B', 'HR', 'TB', 'RBI', 'SAC', 'SF', 'BB', 'IBB', 'HBP', 'SO',
       'GDP', 'SLG', 'OBP', 'OPS', 'MH', 'RISP', 'PH-BA', 'year'],
      dtype='object')
Index(['rank', 'name', 'team', 'ERA', 'G', 'W', 'L', 'SV', 'HLB', 'WPCT', 'IP',
       'H', 'HR', 'BB', 'HBP', 'SO', 'R', 'ER', 'WHIP', 'CG', 'SHO', 'QS',
       'BSV', 'TBF', 'NP', 'AVG', '2B', '3B', 'SAC', 'SF', 'IBB', 'WP', 'BK',
       'year'],
      dtype='object')
Index(['rank', 'playerId', 'name', 'team', 'AVG', 'G', 'PA', 'AB', 'R', 'H',
       '2B', '3B', 'HR', 'TB', 'RBI', 'SAC', 'SF', 'year', 'teamCode',
       'teamName', 'SB', 'CS', 'BB', 'HBP', 'SO', 'GDP', 'E',
       'is_goldenglove'],
      dtype='object')
Index(['rank', 'playerId', 'name', 'team', 'ERA', 'G', 'W', 'L', 'SV', 'HLD',
       'WPCT', 'IP', 'H', 'HR', 'BB', 'HBP', 'SO', 'R', 'ER', 'WHIP', 'year',
       'teamCode', 'teamName', 'CG', 'SHO', 'TBF', 'is_goldenglove'],
  

In [31]:
# 2015-2025 데이터: layerId, teamCode, teamName 삭제
# 결측치 데이터 확인 후 평균 값 or 삭제

### 2. 골든글러브 데이터 전처리
#### 2-1. 칼럼명 정리
year, P, C, 1B, 2B, 3B, SS, OF1, OF2, OF3, OF4, DH(기존) --> year, position, player_team(전처리 후)

#### 2-2. 선수명/팀명 분리
현재 데이터에는 `폰세 한화`로 되어있어서 분리가 필요하다.

#### 2-3. 2015-2025 학습 데이터에 `is_goldenglove` 라벨 추가

In [51]:
# 2-1. 칼럼 및 데이터 정리
position_cols = ["P", "C", "1B", "2B", "3B", "SS", "OF1", "OF2", "OF3", "OF4", "DH"]

golden = golden_glove.melt(
    id_vars = "year",
    value_vars = position_cols,
    var_name = "position",
    value_name = "player_name"
)

# 빈 값 제거
golden = golden.dropna(subset=["player_name"])

# OF1..4 -> OF로 통일
golden["position"] = golden["position"].replace({
    "OF1": "OF",
    "OF2": "OF",
    "OF3": "OF",
    "OF4": "OF"
})

# 선수명, 팀이름 분리
golden[["name", 'team']] = golden["player_name"].str.rsplit(" ", n = 1, expand = True)

# Strip whitespace after splitting
golden["name"] = golden["name"].str.strip()
golden["team"] = golden["team"].str.strip()

golden = golden[["year", "position", "name", "team"]]

# Rename columns to match Kaggle data's capitalization for merging/matching later
golden = golden.rename(columns={"year": "Year", "name": "Name", "team": "Team"})

golden.head()

,Year,position,Name,Team
0,2025,P,폰세,한화
1,2024,P,하트,NC
2,2023,P,페디,NC
3,2022,P,안우진,키움
4,2021,P,미란다,두산


In [35]:
# 학습용 kaggle 데이터에 is_goldenglove 라벨 붙이기
hitter_recent["is_goldenglove"] = hitter_recent.set_index(["year", "name", "team"]).index.isin(
    golden.set_index(["Year", "Name", "Team"]).index
).astype(int)

pitcher_recent["is_goldenglove"] = pitcher_recent.set_index(["year", "name", "team"]).index.isin(
  golden.set_index(["Year", "Name", "Team"]).index
).astype(int)

hitter_recent.head()
pitcher_recent.head()

# is_goldenglove 라벨 확인: 골든글러브 수상자면 1, 아니면 0
hitter_recent[hitter_recent["name"] == "김도영"].head()


,rank,playerId,name,team,AVG,G,PA,AB,R,H,...,teamCode,teamName,SB,CS,BB,HBP,SO,GDP,E,is_goldenglove
272,3,52605,김도영,KIA,0.309,30,122,110,20,34,...,HT,KIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
400,3,52605,김도영,KIA,0.347,141,625,544,143,189,...,HT,KIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
964,4,52605,김도영,KIA,0.303,84,385,340,72,103,...,HT,KIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1311,16,52605,김도영,KIA,0.237,103,254,224,37,53,...,HT,KIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [37]:
# 라벨이 잘 붙었는지 확인

print(hitter_recent["is_goldenglove"].value_counts())
print(pitcher_recent["is_goldenglove"].value_counts())

print(hitter_recent.groupby("year")["is_goldenglove"].sum())
print(pitcher_recent.groupby("year")["is_goldenglove"].sum())

is_goldenglove
0    3610
1      85
Name: count, dtype: int64
is_goldenglove
0    2724
1      11
Name: count, dtype: int64
year
2016    8
2017    8
2018    8
2019    9
2020    9
2021    9
2022    8
2023    9
2024    9
2025    8
Name: is_goldenglove, dtype: int64
year
2016    1
2017    1
2018    1
2019    1
2020    1
2021    1
2022    2
2023    1
2024    1
2025    1
Name: is_goldenglove, dtype: int64


In [71]:
# 팀 이적으로 인한 골든글러브 라벨 생성 실패 건 확인
# 골든글러브 수상자 팀명 보정
# 기준: 시상식 당시 소속이 아니라 해당 시즌 정규시즌 기록 데이터의 Team 값

team_corrections = {
    (2017, "강민호"): "롯데",
    (2016, "최형우"): "삼성",
    (2022, "양의지"): "NC",
    (2025, "최형우"): "KIA"
}

for (year, name), team in team_corrections.items():
    golden.loc[
        (golden["Year"] == year) & (golden["Name"] == name),
        "Team"
    ] = team

# 2018 이정후 선수의 Name, Team 분리 안되는 문제 해결
golden.loc[
    (golden["Year"] == 2018) &
    (golden["Team"].str.contains("이정후", na=False)),
    ["Name", "Team"]
] = ["이정후", "넥센"]


In [75]:
# golden_hitter_recent 생성 후 매칭 확인
hitter_positions = ["C", "1B", "2B", "3B", "SS", "OF", "DH"]

golden_hitter_recent = golden[
    (golden["position"].isin(hitter_positions)) &
    (golden["Year"] >= 2016) &
    (golden["Year"] <= 2025)
].copy()

hitter_keys = set(zip(hitter_recent["year"], hitter_recent["name"], hitter_recent["team"]))

golden_hitter_recent["matched"] = golden_hitter_recent.apply(
    lambda row: (row["Year"], row["Name"], row["Team"]) in hitter_keys,
    axis=1
)

failed_hitter = golden_hitter_recent[golden_hitter_recent["matched"] == False]

print("매칭 실패 인원:", len(failed_hitter))
display(failed_hitter)

매칭 실패 인원: 0


,Year,position,Name,Team,matched


In [23]:
# 세부지표 NaN값 확인
print(hitter.columns.tolist())
print(pitcher.columns.tolist())

# 결측치 비율 확인(NaN)
hitter.isna().mean().sort_values(ascending=False).head(30)
pitcher.isna().mean().sort_values(ascending=False).head(30)

['순위', 'playerId', '선수명', '팀명', 'AVG', 'G', 'PA', 'AB', 'R', 'H', '2B', '3B', 'HR', 'TB', 'RBI', 'SAC', 'SF', 'year', 'teamCode', 'teamName', 'SB', 'CS', 'BB', 'HBP', 'SO', 'GDP', 'E']
['순위', 'playerId', '선수명', '팀명', 'ERA', 'G', 'W', 'L', 'SV', 'HLD', 'WPCT', 'IP', 'H', 'HR', 'BB', 'HBP', 'SO', 'R', 'ER', 'WHIP', 'year', 'teamCode', 'teamName', 'CG', 'SHO', 'TBF']


,0
CG,0.94224
TBF,0.94224
SHO,0.94224
WHIP,0.05776
ERA,0.00000
G,0.00000
선수명,0.00000
팀명,0.00000
순위,0.00000
playerId,0.00000


In [57]:
# hitter_recent 및 pitcher_recent 데이터프레임의 결측치 비율 확인
print("\n--- hitter_recent Missing Values ---")
print(hitter_recent.isna().mean().sort_values(ascending=False).head(30))

print("\n--- pitcher_recent Missing Values ---")
print(pitcher_recent.isna().mean().sort_values(ascending=False).head(30))


--- hitter_recent Missing Values ---
SO                1.0
GDP               1.0
E                 1.0
HBP               1.0
BB                1.0
SB                1.0
CS                1.0
AVG               0.0
rank              0.0
playerId          0.0
name              0.0
team              0.0
3B                0.0
2B                0.0
H                 0.0
R                 0.0
AB                0.0
PA                0.0
G                 0.0
HR                0.0
teamName          0.0
teamCode          0.0
year              0.0
SF                0.0
SAC               0.0
RBI               0.0
TB                0.0
is_goldenglove    0.0
dtype: float64

--- pitcher_recent Missing Values ---
SHO               1.0
CG                1.0
TBF               1.0
team              0.0
ERA               0.0
G                 0.0
W                 0.0
L                 0.0
rank              0.0
playerId          0.0
name              0.0
WPCT              0.0
HLD               0.0
SV    

In [56]:
# hitter_recent 및 pitcher_recent 데이터프레임의 결측치 비율 확인
print("\n--- hitter_recent Missing Values ---")
print(hitter_recent.isna().mean().sort_values(ascending=False).head(30))

print("\n--- pitcher_recent Missing Values ---")
print(pitcher_recent.isna().mean().sort_values(ascending=False).head(30))


--- hitter_recent Missing Values ---
SO                1.0
GDP               1.0
E                 1.0
HBP               1.0
BB                1.0
SB                1.0
CS                1.0
AVG               0.0
rank              0.0
playerId          0.0
name              0.0
team              0.0
3B                0.0
2B                0.0
H                 0.0
R                 0.0
AB                0.0
PA                0.0
G                 0.0
HR                0.0
teamName          0.0
teamCode          0.0
year              0.0
SF                0.0
SAC               0.0
RBI               0.0
TB                0.0
is_goldenglove    0.0
dtype: float64

--- pitcher_recent Missing Values ---
SHO               1.0
CG                1.0
TBF               1.0
team              0.0
ERA               0.0
G                 0.0
W                 0.0
L                 0.0
rank              0.0
playerId          0.0
name              0.0
WPCT              0.0
HLD               0.0
SV    